# Week 14 — Voice Agent: LiveKit + LangGraph Integration
_Wiring the standalone voice pipeline (Notebook 03) into the multi-agent orchestrator_

---

Notebook 03 took apart STT, TTS, VAD, and EOU as separate pieces. This
notebook wires them into the **Week 13 multi-agent orchestrator** so the
user can talk to the hospital assistant by voice.

## Architecture

```
User browser ──── audio ────▶ LiveKit Cloud Room ────▶ Voice Worker (our process)
                                                          │
                                            ┌─────────────┴────────────┐
                                            │  Silero VAD              │
                                            │       │ 32 ms frames     │
                                            │       ▼                  │
                                            │  Deepgram STT (Nova-3)   │
                                            │       │ transcript       │
                                            │       ▼                  │
                                            │  LangGraphLLMAdapter ────┼──▶ orchestrator.achat()
                                            │       │ answer text      │           │
                                            │       ▼                  │           ▼
                                            │  ElevenLabs TTS (Turbo v2.5)    │   recall→supervisor→agents
                                            │                          │   →merge→save
                                            └─────────────┬────────────┘
                                                          │ audio
User browser ◀─── audio ─── LiveKit Cloud Room ◀──────────┘
```

## What you'll learn

| Section | Topic |
|---------|-------|
| 1 | The adapter pattern — how voice plugs into the orchestrator |
| 2 | Voice configuration (`voice:` section in `param.yaml`) |
| 3 | The agent factory — how Silero/Deepgram/Adapter become a `livekit.agents.voice.Agent` |
| 4 | Running the worker (native + Docker) |
| 5 | Barge-in — what happens when the user interrupts |
| 6 | Latency budget for the full voice loop |
| 7 | Demo script — four spoken queries that exercise each agent route |

## Note on the integration depth

**Voice goes through `orchestrator.achat()`**, which uses the legacy
multi-agent graph (`recall → supervisor → agents → merge → save`).
The HTTP `/chat` endpoint goes through the newer **decision graph**
first (parallel guardrail + router + CAG cache) and only falls through
to the multi-agent graph if the verdict is `proceed`.

Voice users therefore *don't* get the guardrail / CAG short-circuits
yet. That's intentional — the voice module is fully decoupled from
`api/routers/chat.py` so it can ship and evolve independently. Lifting
voice onto the decision graph is a Week 15 task.

---

## Section 0 — Setup

In [1]:
# Install (or top up) every package this notebook uses, into the kernel
# that's actually running. `%pip` is the kernel-aware magic — `!pip` may
# install somewhere else if you launched Jupyter from a different Python.
# Re-running this cell is a no-op once the deps are present.
%pip install --quiet \
    python-dotenv \
    deepgram-sdk>=6.0.0 \
    elevenlabs>=2.0.0 \
    sounddevice>=0.5.0 \
    ipywidgets>=8.1.1 \
    scipy>=1.11.0 \
    numpy>=1.26.0 \
    livekit-agents>=1.5.0 \
    livekit-plugins-deepgram>=1.5.0 \
    livekit-plugins-elevenlabs>=1.5.0 \
    livekit-plugins-silero>=1.5.0
print('Notebook 04 deps OK ✓')


zsh:1: 6.0.0 not found
Note: you may need to restart the kernel to use updated packages.
Notebook 04 deps OK ✓


In [2]:
import os, sys, time, asyncio
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv()

import nest_asyncio
nest_asyncio.apply()

from infrastructure.log import setup_logging
setup_logging('INFO', for_notebook=True)
from loguru import logger
logger.info('notebook ready')

2026-05-10 18:13:55.115 | DEBUG    | infrastructure.observability:<module>:209 - langfuse package not installed — @observe is a no-op.


ℹ️ notebook ready


---
## Section 1 — The adapter pattern

LiveKit's `Agent` expects an `LLM` plugin. Our "LLM" is actually a
whole multi-agent graph. The adapter implements LiveKit's `LLM`
interface but, instead of calling an OpenAI completion endpoint, it
calls `orchestrator.achat()`.

```python
# What LiveKit thinks it's getting
Agent(instructions=..., stt=stt, llm=adapter, tts=tts, vad=vad)

# What the adapter actually does
class LangGraphLLMStream(LLMStream):
    async def _run(self):
        text = chat_ctx.messages[-1].content              # STT transcript
        response = await orchestrator.achat(text, ...)    # full graph
        emit(ChatChunk(delta=ChoiceDelta(content=response.answer)))
```

Let's exercise the adapter directly — *without* LiveKit — to prove the
wiring works.

In [3]:
from agents.orchestrator import build_agent
from voice.adapter import LangGraphLLMAdapter

orchestrator = build_agent(enable_crm=True, enable_rag=True, enable_web=True)
adapter = LangGraphLLMAdapter(
    orchestrator=orchestrator,
    user_id='nb-04-demo',
    session_id='nb-04-session',
)
logger.success(f'adapter ready (user={adapter._user_id})')

ℹ️ ✓ Supabase SQL engine created
ℹ️ ✅ Supabase connection test: SUCCESS
ℹ️ ✅ pgvector extension: INSTALLED
ℹ️ ✓ Schema validation passed: vector(1536)
ℹ️ CRM tool initialised
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
ℹ️ Connected to Qdrant Cloud at https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"
ℹ️ ✓ CAG cache ready (Qdrant collection='cag_cache', dim=1536, threshold=0.65)
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections/cag_cache "HTTP/1.1 200 OK"
ℹ️ RAGTool initialised: CAG cache (CAGCache(collection='cag_cache', threshold=0.65, ttl=86400s, entries=94, backend='qdrant')) -> CRAG (k=4, expanded_k=8, threshold=0.60)
ℹ️ RAG tool initialised
ℹ️ Web search tool initialised
✅ adapter read

In [4]:
# Drive the adapter manually with a synthetic chat context — same shape
# LiveKit produces after STT finalises.
from livekit.agents.llm import ChatContext

async def adapter_turn(text: str) -> str:
    """Feed `text` into the adapter and return the answer."""
    ctx = ChatContext()
    ctx.add_message(role='user', content=text)
    stream = adapter.chat(chat_ctx=ctx)
    chunks = []
    async for chunk in stream:
        if chunk.delta and chunk.delta.content:
            chunks.append(chunk.delta.content)
    return ''.join(chunks)

# Direct route — should be fast, no tools.
t0 = time.perf_counter()
answer = await adapter_turn('Hello, my name is Anushka.')
ms = int((time.perf_counter() - t0) * 1000)
print(f'[{ms} ms] {answer[:300]}')

ℹ️ Voice → Agent: "Hello, my name is Anushka."
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ Retrieved 5 facts from LT memory for user nb-04-demo
ℹ️ Recalled 4 ST turns, 3 LT facts for user nb-04-demo
ℹ️ HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ Distilling new facts for nb-04-demo...
ℹ️ HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ Upserted 5 facts to LT memory (0 new, 5 merged)
ℹ️ Distilled 5 facts for user nb-04-demo
✅ Agent → TTS: "Hello Anushka! It's a pleasure to meet you. How can I assist you today at Nawalo..." [route=direct, 19004 ms]
[19008 ms] Hello Anushka! It's a pleasure to meet you. How can I assist you toda

In [5]:
# CRM route — touches the database.
t0 = time.perf_counter()
answer = await adapter_turn('Which cardiologists are available at Nawaloka?')
ms = int((time.perf_counter() - t0) * 1000)
print(f'[{ms} ms] {answer[:400]}')

ℹ️ Voice → Agent: "Which cardiologists are available at Nawaloka?"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ Retrieved 5 facts from LT memory for user nb-04-demo
ℹ️ Recalled 4 ST turns, 3 LT facts for user nb-04-demo
ℹ️ HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ Distilling new facts for nb-04-demo...
ℹ️ HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ Upserted 9 facts to LT memory (0 new, 9 merged)
ℹ️ Distilled 9 facts for user nb-04-demo
✅ Agent → TTS: "Anushka, here are the cardiologists available at Nawaloka Hospitals:

*   **Dr. ..." [route=crm, 20690 ms]
[20692 ms] Anushka, here are the cardiologists available at Naw

In [6]:
# RAG route — pulls from the internal knowledge base.
t0 = time.perf_counter()
answer = await adapter_turn('What cardiac diagnostic services does the hospital offer?')
ms = int((time.perf_counter() - t0) * 1000)
print(f'[{ms} ms] {answer[:400]}')

ℹ️ Voice → Agent: "What cardiac diagnostic services does the hospital offer?"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ Retrieved 5 facts from LT memory for user nb-04-demo
ℹ️ Recalled 4 ST turns, 3 LT facts for user nb-04-demo
ℹ️ HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections/cag_cache/points/query "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections/nawaloka/points/query "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://openrouter.ai/api/v1/embeddings "HTTP/1.1 200 OK"
ℹ️ HTTP Request: POST https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.q

Three different routes, same adapter, same orchestrator. The voice
layer doesn't have to know anything about routing — it just hands
transcripts to `achat()` and pipes the answer back to TTS.

---
## Section 2 — Voice configuration

Everything tunable lives in `config/param.yaml` under `voice:`.
Credentials come from `.env`. Let's see what's currently loaded.

In [7]:
from voice.config import load_voice_config

cfg = load_voice_config()
print('Voice configuration')
print('=' * 60)
print(f'STT          : {cfg.stt_provider}/{cfg.stt_model}  (lang={cfg.stt_language})')
print(f'TTS          : {cfg.tts_provider}/{cfg.tts_model}')
if cfg.tts_provider == 'elevenlabs':
    print(f'TTS voice ID : {cfg.tts_voice_id}')
print(f'VAD threshold: {cfg.vad_threshold}')
print(f'Silence (L2) : {cfg.silence_threshold_ms} ms')
print(f'Endpoint (L3): {cfg.min_endpointing_delay} s')
print(f'Interrupts   : {cfg.interruption_enabled}')
print(f'Sample rate  : {cfg.sample_rate} Hz')
print()
print(f'LIVEKIT_URL  : {cfg.livekit_url or "(unset)"}')
print(f'LiveKit creds: {bool(cfg.livekit_api_key) and bool(cfg.livekit_api_secret)}')
print(f'Deepgram key : {bool(cfg.deepgram_api_key)}  (STT)')
print(f'ElevenLabs   : {bool(cfg.eleven_api_key)}  (TTS — required if tts_provider=elevenlabs)')

Voice configuration
STT          : deepgram/nova-3  (lang=en)
TTS          : elevenlabs/eleven_turbo_v2_5
TTS voice ID : l7kNoIfnJKPg7779LI2t
VAD threshold: 0.5
Silence (L2) : 500 ms
Endpoint (L3): 0.5 s
Interrupts   : True
Sample rate  : 16000 Hz

LIVEKIT_URL  : wss://aee-bootcamp-4yz38pau.livekit.cloud
LiveKit creds: True
Deepgram key : True  (STT)
ElevenLabs   : True  (TTS — required if tts_provider=elevenlabs)


### Tuning guide (recap from Notebook 03 §4)

| Knob | Default | Make it bigger ⇒ | Make it smaller ⇒ |
|------|--------:|------------------|-------------------|
| `vad_threshold` | 0.5 | Misses quiet/whispered speech | More noise triggers |
| `silence_threshold_ms` | 500 | Tolerates longer pauses | Fires on every breath |
| `min_endpointing_delay` | 0.5 s | More patient | Faster but may cut off |

For Nawaloka (mixed English / Sinhala / Tamil speakers, often elderly),
the defaults are tuned slightly conservative. If you ship a fast-paced
support assistant instead, drop silence to 300 ms and endpointing to 0.3 s.

---
## Section 3 — The agent factory

`build_voice_agent()` in `src/voice/agent.py` is where the four pieces
come together. Walk through it line by line — the comments are the
lecture.

In [8]:
import inspect
from voice import agent as voice_agent

print(inspect.getsource(voice_agent.build_voice_agent))

def build_voice_agent(
    *,
    participant: rtc.RemoteParticipant,
    room_name: str,
    cfg: VoiceConfig,
    orchestrator: AgentOrchestrator,
) -> tuple[Agent, VoiceSession]:
    """Build a configured LiveKit ``Agent`` for one participant.

    Returns the Agent (for the AgentSession to start) and the VoiceSession
    record (for event handlers to update).
    """
    user_id = participant.identity or "unknown-user"
    session = _session_manager.get_or_create(
        participant_id=participant.identity,
        user_id=user_id,
        room_name=room_name,
    )
    logger.info(f"Participant joined: {user_id} → {session.session_id}")

    # The bridge between voice and the agent.
    adapter = LangGraphLLMAdapter(
        orchestrator=orchestrator,
        user_id=user_id,
        session_id=session.session_id,
    )

    # Plugins.
    vad = silero.VAD.load(
        min_silence_duration=cfg.silence_threshold_ms / 1000.0,
        activation_threshold=cfg.vad_threshold,
       

**Three things to notice:**

1. `silero.VAD.load(min_silence_duration=cfg.silence_threshold_ms / 1000.0, ...)`
   — this is where Notebook 03's EOU layer-2 parameter actually lives in
   production code. The number you saw in the parameter playground is
   the same number that gets passed here.
2. `Agent(...)` is stateless. It's a *definition*. The state lives in
   `AgentSession`, which is started per-room.
3. We stash the adapter on the Agent object (`agent._lg_adapter = adapter`)
   so barge-in event handlers can call `adapter.cancel_current()` later.

---
## Section 4 — Running the worker

There are two ways to run the voice worker. Both connect outbound to
**LiveKit Cloud** (this week — local LiveKit server is Week 15).

### Mode A — Native (development)

```bash
# 1. Make sure .env has LIVEKIT_* and DEEPGRAM_API_KEY filled in.
# 2. Run the worker in foreground:
make voice
```

Expected output:

```
  Nawaloka Hospital — Voice Worker
  ------------------------------------------------
  LiveKit URL  : wss://your-project.livekit.cloud
  STT          : deepgram/nova-3
  TTS          : deepgram/aura-2-andromeda-en
  ...
  ------------------------------------------------

[INFO] registered worker id=...  region=...
[INFO] waiting for jobs
```

### Mode B — Docker (production-shaped)

```bash
make demo-voice
# brings up: api (8000), web (8080), voice (no port — outbound only)

make voice-logs
# tails the voice container's stdout
```

The `voice` service is opt-in via a Compose **profile** — `make demo`
alone still starts only `api` and `web`, untouched.

### Connecting a browser to talk

Once the worker is running, open
[**LiveKit Agents Playground**](https://agents-playground.livekit.io),
paste your LiveKit project URL + an access token, and click **Connect**.
The worker will pick up the room and greet you.

---
## Section 5 — Barge-in mechanics

Barge-in is the test of whether a voice agent feels human.

```
Time →
Agent: "Dr. Perera is available on Monday and Wednes—"
User :                                              "Actually, I want a dentist."
                                                     │
                                                     ├─ VAD detects speech mid-TTS
                                                     ├─ Agent TTS stops immediately
                                                     ├─ adapter.cancel_current()
                                                     │   cancels the in-flight achat() task
                                                     └─ New STT begins with the new utterance
```

Three things have to coordinate:

1. **VAD keeps running while the agent speaks.** That's a LiveKit
   default — `Agent(allow_interruptions=True)` enables continuous VAD.
2. **TTS is cancellable mid-chunk.** LiveKit cuts the audio output
   when the speech-interrupted event fires.
3. **The in-flight LLM task gets cancelled.** That's our adapter's
   `cancel_current()`. Without it, the orchestrator would finish its
   work and write a stale answer to memory.

Look at the wiring in `agent.py`:

In [9]:
# Pull just the barge-in handler section out of agent.py for inspection.
src = inspect.getsource(voice_agent.create_and_start_agent)
for line in src.splitlines():
    if 'interrupt' in line.lower() or 'cancel' in line.lower():
        print(line)

    @agent_session.on("agent_false_interruption")
    def _on_false_interrupt(_ev):
    # Cancel in-flight agent task on barge-in.
        @agent_session.on("agent_speech_interrupted")  # type: ignore[arg-type]
        def _on_interrupted(_ev):
            on_agent_speech_interrupted(session)
            adapter.cancel_current()
        # state-change driven cancellation only.
        logger.debug("agent_speech_interrupted event not available; using state changes")
        allow_interruptions=True,


---
## Section 6 — Latency budget

End-to-end latency = time from "user stops speaking" to "first audio
byte hits the user's speaker".

| Stage | Where | Typical | Tunable? |
|-------|-------|--------:|----------|
| EOU (silence + endpointing) | Silero + LiveKit | ~500 ms + 500 ms | Yes — `silence_threshold_ms`, `min_endpointing_delay` |
| STT finalisation | Deepgram (streaming) | ~300 ms | Mostly fixed |
| Agent | `orchestrator.achat()` | 1–3 s (route-dependent) | Yes — caching, faster LLM, shorter prompts |
| TTS time-to-first-byte | ElevenLabs Turbo v2.5 | ~250 ms | Mostly fixed |
| WebRTC transport | LiveKit Cloud | ~50 ms | Mostly fixed |
| **End-to-end** | | **~3–5 s** | |

Per-route agent latencies (measured from the adapter calls in §1):

| Route | What runs | Typical |
|-------|-----------|--------:|
| Direct (greeting) | 1 LLM call | ~800 ms |
| Web search | LLM + Tavily | ~1.5 s |
| CRM | LLM + Supabase SQL | ~2 s |
| RAG | LLM + Qdrant + LLM synth | ~2.5 s |
| Multi-route | Both fanned out + merge | ~3 s |

**Where to optimise next:**

- The biggest single cost is `achat()` itself. Lifting voice onto the
  decision graph (Week 15) lets us short-circuit common queries via
  the CAG cache (~30 ms).
- Drop `silence_threshold_ms` to 300 ms once you've measured your
  callers' typical pause length.
- Stream the agent's response token-by-token instead of as one chunk.
  That requires changing `LangGraphLLMAdapter._run` to use
  `astream` and yield chunks as they arrive — non-trivial because
  the multi-agent graph emits the final answer atomically.

---
## Section 7 — Demo script

Once `make voice` is running and you're connected via the LiveKit
Playground, try these four spoken queries. They each hit a different
agent route.

| # | Say this | Expected route | What the agent should do |
|---|----------|---------------|--------------------------|
| 1 | *"Hello, my name is Anushka."* | direct | Warm greeting, acknowledges your name |
| 2 | *"Which cardiologists are available?"* | crm/search_doctors | Lists doctors from Supabase |
| 3 | *"What cardiac diagnostic services does the hospital offer?"* | rag | Pulls from the internal KB |
| 4 | *"Do I have any appointments this week?"* | crm/lookup_patient | Looks up your bookings |

Bonus tests:

- **Barge-in**: while the agent answers query #2, interrupt with *"Actually, I want a dentist."* — the agent's TTS should cut off mid-word.
- **Memory**: after query #1, ask *"What's my name?"* — should recall "Anushka".
- **EOU tuning**: drop `silence_threshold_ms` to 250 in `param.yaml`, restart `make voice`, and notice how much snappier (and how much more often it cuts you off) the agent feels.


---
## Section 8 — Try it yourself (without LiveKit)

You don't need to spin up a LiveKit room to feel the pipeline.
The cell below records from your microphone, ships the audio to
Deepgram STT, sends the transcript to **the same `orchestrator.achat()`**
the voice agent uses, and plays the answer back through ElevenLabs TTS.

It's the *exact same call path* the LiveKit Agent makes — just driven
from a notebook instead of a WebRTC room. Use it to:

- Sanity-check that your mic, STT key, and TTS key all work end-to-end.
- Compare a route's spoken latency against the table in §6.
- Test new prompts before wiring them through the LiveKit worker.

**What this cell does NOT do:**

- No VAD-based end-of-utterance — you press *Stop* manually.
- No barge-in — TTS plays through to completion.
- No streaming — STT and TTS both run as one-shot HTTP calls.

For all of those, you need the actual LiveKit worker (`make voice`).


In [10]:
# Record → STT → orchestrator.achat() → TTS → playback
# ----------------------------------------------------
# Press Start, speak a query, press Stop. The agent will reply in voice.
#
# Implementation notes:
#  - The pipeline runs as an asyncio background task (not run_until_complete),
#    so the Stop click returns immediately and the UI stays responsive while
#    STT / agent / TTS run.
#  - Deepgram and ElevenLabs ship blocking sync clients in this version; we
#    wrap their calls in asyncio.to_thread() to keep the kernel's event loop
#    free.
#  - We render into the Output widget via output.append_stdout() and
#    output.append_display_data() — NOT `with output:` + print/display. The
#    `with` form doesn't survive async `await` switches reliably; the append
#    methods do.
import asyncio
import io
import os
import time
import traceback

import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
import ipywidgets as widgets
from IPython.display import Audio

from deepgram import DeepgramClient
from elevenlabs.client import ElevenLabs
from agents.orchestrator import build_agent
from voice.config import load_voice_config

cfg          = load_voice_config()
orchestrator = build_agent()
dg           = DeepgramClient(api_key=os.getenv("DEEPGRAM_API_KEY"))
eleven       = ElevenLabs(api_key=os.getenv("ELEVEN_API_KEY"))

SAMPLE_RATE = 16_000
# Use a phone number from the seeded CRM (scripts/seed_crm_unified.py).
# The orchestrator passes user_id through as patient_id for CRM ops, so
# this needs to map to a real row in the patients table or bookings will
# fail with "Patient not found". 94781030736 is the standard demo seed.
# Use a phone (external_user_id) from the seeded CRM. The CRM tool
# resolves this to the canonical patients.patient_id internally, so the
# booking's foreign key lands on the right row.
#   94726786211  →  Anushka Perera
#   94761070189  →  Sunil Fernando (asthmatic)
#   94751019678  →  Kamal Jayasuriya (hypertensive)
USER_ID     = "94726786211"
SESSION_ID  = "notebook-voice-demo"

# --- Recording state ---------------------------------------------------------
_frames: list[np.ndarray] = []
_stream: sd.InputStream | None = None
_task:   asyncio.Task | None = None

def _on_audio(indata, frames, time_info, status_flag):
    _frames.append(indata.copy())

# --- UI ---------------------------------------------------------------------
btn_start = widgets.Button(description="● Start", button_style="danger")
btn_stop  = widgets.Button(description="■ Stop",  button_style="",       disabled=True)
status    = widgets.HTML("<i>Idle — press Start to record.</i>")
output    = widgets.Output()

def start_recording(_):
    global _stream
    _frames.clear()
    _stream = sd.InputStream(samplerate=SAMPLE_RATE, channels=1,
                             dtype="int16", callback=_on_audio)
    _stream.start()
    btn_start.disabled = True
    btn_stop.disabled  = False
    status.value = "<b style=\"color:#c00\">● Recording… speak now.</b>"

def stop_recording(_):
    global _stream, _task
    if _stream is not None:
        try:
            _stream.stop(); _stream.close()
        except Exception as exc:
            output.append_stderr(f"[warn] stream close: {exc!r}\n")
        _stream = None

    btn_start.disabled = False
    btn_stop.disabled  = True

    if not _frames:
        status.value = "<i>Nothing captured — try again.</i>"
        return

    audio = np.concatenate(_frames, axis=0)
    status.value = "<b>⏳ Processing… (STT → agent → TTS, ~3–5 s)</b>"
    output.clear_output(wait=True)
    _task = asyncio.ensure_future(_run_pipeline(audio))

async def _run_pipeline(audio: np.ndarray) -> None:
    try:
        # 1) WAV bytes in memory
        buf = io.BytesIO()
        wavfile.write(buf, SAMPLE_RATE, audio)
        wav_bytes = buf.getvalue()

        # 2) STT — sync SDK call, push to a thread
        status.value = "<b>① Transcribing…</b>"
        t0 = time.perf_counter()
        stt_resp = await asyncio.to_thread(
            lambda: dg.listen.v1.media.transcribe_file(
                request=wav_bytes,
                model=cfg.stt_model,
                smart_format=True,
            )
        )
        transcript = stt_resp.results.channels[0].alternatives[0].transcript.strip()
        t_stt = time.perf_counter() - t0
        output.append_stdout(f"You said : {transcript!r}   ({t_stt*1000:.0f} ms STT)\n")
        if not transcript:
            status.value = "<i>No speech detected.</i>"
            return

        # 3) Agent — same call path the LiveKit adapter uses
        status.value = "<b>② Asking the agent…</b>"
        t0 = time.perf_counter()
        response = await orchestrator.achat(
            user_message=transcript,
            user_id=USER_ID,
            session_id=SESSION_ID,
        )
        t_agent = time.perf_counter() - t0
        output.append_stdout(f"Agent    : {response.answer!r}   ({t_agent:.2f} s achat)\n")
        output.append_stdout(f"Route    : {getattr(response, 'route', 'unknown')}\n")

        # 4) TTS — also sync, also to a thread
        status.value = "<b>③ Synthesising…</b>"
        t0 = time.perf_counter()
        audio_bytes = await asyncio.to_thread(
            lambda: b"".join(eleven.text_to_speech.convert(
                voice_id=cfg.tts_voice_id,
                model_id=cfg.tts_model,
                text=response.answer,
                output_format="mp3_44100_128",
            ))
        )
        t_tts = time.perf_counter() - t0
        output.append_stdout(f"TTS bytes: {len(audio_bytes):,}   ({t_tts*1000:.0f} ms)\n")
        output.append_stdout(f"Total    : {(t_stt + t_agent + t_tts):.2f} s end-to-end\n")

        # 5) Playback — append_display_data is async-safe; `display()` is not
        status.value = "<b>✓ Done — playing reply.</b>"
        output.append_display_data(Audio(audio_bytes, autoplay=True))

    except Exception:
        output.append_stderr("Pipeline error:\n")
        output.append_stderr(traceback.format_exc())
        status.value = "<i style=\"color:#c00\">Error — see output above.</i>"

btn_start.on_click(start_recording)
btn_stop.on_click(stop_recording)
display(widgets.HBox([btn_start, btn_stop]), status, output)


ℹ️ CRM tool initialised
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections "HTTP/1.1 200 OK"
ℹ️ ✓ CAG cache ready (Qdrant collection='cag_cache', dim=1536, threshold=0.65)
ℹ️ HTTP Request: GET https://025872ed-03a9-42c8-84ff-5caad58a460b.us-east-1-1.aws.cloud.qdrant.io:6333/collections/cag_cache "HTTP/1.1 200 OK"
ℹ️ RAGTool initialised: CAG cache (CAGCache(collection='cag_cache', threshold=0.65, ttl=86400s, entries=94, backend='qdrant')) -> CRAG (k=4, expanded_k=8, threshold=0.60)
ℹ️ RAG tool initialised
ℹ️ Web search tool initialised


HTML(value='<i>Idle — press Start to record.</i>')

Output()

---
## Summary

1. **The adapter is the only point of contact** between voice and the
   agent. Everything else is generic LiveKit plumbing.
2. **`achat()` is the bridge.** Voice doesn't go through `chat.py`'s
   decision graph (yet) — that's a deliberate decoupling so voice can
   evolve independently.
3. **Barge-in is three coordinated cancellations**: VAD detects, TTS
   stops, agent task gets cancelled.
4. **Latency is dominated by the agent**, not by STT/TTS. Optimisations
   should focus there.
5. **`make voice` (native) and `make demo-voice` (Docker)** are the two
   ways to run the worker. Pick native for development, Docker for
   anything that talks to a real customer.
6. **Section 8's recording widget** is the fastest way to test a new
   prompt or a new orchestrator route — same call path, no LiveKit
   server required.
